# Active OPD Explorer

This notebook explores the local Active OPD implementation without downloading Qwen checkpoints by default. It includes a CPU/CUDA tiny-model trainer smoke test, lazy OpenThoughts/MATH inspection, auditable answer filtering, and an explicitly guarded real-Qwen smoke entry point.

In [ ]:
from pathlib import Path
import sys

# Make the repository importable from a notebook launched in notebooks/ or the repo root.
REPO_ROOT = Path.cwd()
if REPO_ROOT.name == "notebooks":
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import torch
from torch import nn
import matplotlib.pyplot as plt

from aopd.data import (
    MathExample,
    OpenThoughtsConfig,
    OpenThoughtsDataset,
    Rollout,
    example_from_record,
    extract_final_answer,
    load_math500,
)
from aopd.losses import OPDLossConfig, response_token_mask
from aopd.models import GenerationOptions
from aopd.train import OPDTrainer, TrainerConfig, VerifiedWrongSelector

torch.manual_seed(7)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"device: {device}")
print(f"torch: {torch.__version__}")

# These flags are deliberately false: defining the helpers below performs no
# network or model I/O. Flip a flag only in a cell where that I/O is intended.
ALLOW_REMOTE_DATA = False
RUN_REAL_MODEL_SMOKE = False

In [ ]:
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(device)
    print(f"GPU: {props.name}")
    print(f"VRAM: {props.total_memory / 2**30:.2f} GiB")
    print(f"CUDA runtime: {torch.version.cuda}")
else:
    print("CUDA is unavailable; the same smoke test will run on CPU.")

## Lazy OpenThoughts and MATH inspection

The dataset adapters do not call `datasets.load_dataset` until iteration. The preview helper below is safe to define or import; pass `allow_remote=True` only when you intentionally want to read a small streaming sample.

In [ ]:
def preview_examples(source, *, name, limit=3, allow_remote=False):
    """Print prompt/reference pairs from a bounded lazy source."""
    if not allow_remote:
        print(f"{name}: remote loading disabled; pass allow_remote=True to opt in.")
        return []

    examples = []
    for index, item in enumerate(source):
        if index >= limit:
            break
        example = item if isinstance(item, MathExample) else example_from_record(item)
        examples.append(example)
        print(f"[{name} {index}] id={example.problem_id}")
        print("prompt:", example.prompt[:500])
        print("reference:", example.reference_answer)
        print()
    return examples

# No data is fetched by this cell. These are explicit, bounded opt-in calls:
# open_thoughts_examples = preview_examples(
#     OpenThoughtsDataset(OpenThoughtsConfig(limit=3)),
#     name="OpenThoughts",
#     allow_remote=ALLOW_REMOTE_DATA,
# )
# math500_examples = preview_examples(
#     (load_math500(streaming=True)),
#     name="MATH-500",
#     allow_remote=ALLOW_REMOTE_DATA,
# )

## Reasoning traces and verified-wrong selection

Rollouts are kept as auditable records: answer extraction is shown separately from exact-match verification, and selected/discarded traces remain inspectable.

In [ ]:
def inspect_rollouts(rollouts, *, trace_chars=700):
    """Extract answers, select verified-wrong traces, and print both pools."""
    records = list(rollouts)
    selector = VerifiedWrongSelector()
    selection = selector.select(records)
    selected_ids = {id(rollout) for rollout in selection.selected}

    print("verification counts:", dict(selection.summary.counts))
    print("selected:")
    for rollout in selection.selected:
        extraction = extract_final_answer(rollout.response)
        print(f"  {rollout.rollout_id}: answer={extraction.answer!r}")
        print(f"    trace: {rollout.response[:trace_chars]}")

    print("discarded:")
    for rollout in records:
        if id(rollout) not in selected_ids:
            result = rollout.verification
            outcome = result.outcome if result is not None else "unverified"
            print(f"  {rollout.rollout_id}: outcome={outcome}")
            print(f"    trace: {rollout.response[:trace_chars]}")
    return selection

example_rollouts = [
    Rollout(
        prompt="What is 2 + 2?",
        response="We add the two terms. Final answer: 4",
        reference_answer="4",
        rollout_id="correct",
    ),
    Rollout(
        prompt="What is 2 + 2?",
        response="A tempting arithmetic slip gives \\boxed{5}.",
        reference_answer="4",
        rollout_id="verified-wrong",
    ),
    Rollout(
        prompt="What is 2 + 2?",
        response="I need more information.",
        reference_answer="4",
        rollout_id="malformed",
    ),
]
selection = inspect_rollouts(example_rollouts)
print("retained for active OPD:", [r.rollout_id for r in selection.selected])

## Optional real-model smoke

The real path uses the configured `StudentModel` (`Qwen/Qwen3-1.7B`), `TeacherModel` (`Qwen/Qwen3-4B`), and custom `OPDTrainer` for one short rollout and one optimizer step. It is disabled even when this notebook is executed top-to-bottom; enable it deliberately with `RUN_REAL_MODEL_SMOKE = True`.

In [ ]:
if RUN_REAL_MODEL_SMOKE:
    import subprocess

    subprocess.run(
        [
            sys.executable,
            "-m",
            "scripts.real_model_smoke",
            "--run",
            "--max-new-tokens",
            "16",
        ],
        cwd=REPO_ROOT,
        check=True,
    )
else:
    print(
        "Real-model smoke disabled. Set RUN_REAL_MODEL_SMOKE = True and "
        "rerun this cell to opt into model downloads and execution."
    )

In [ ]:
class TinyLM(nn.Module):
    def __init__(self, vocab_size=7, hidden_size=12):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, hidden_size)
        self.lm_head = nn.Linear(hidden_size, vocab_size)

    def forward(self, input_ids, attention_mask=None):
        logits = self.lm_head(self.embedding(input_ids))
        return type("Output", (), {"logits": logits})()


class TinyWrapper:
    def __init__(self, model):
        self.model = model

    def prepare_for_training(self):
        self.model.train()
        return self

    def load(self):
        return self

    def forward(self, **inputs):
        return self.model(**inputs)


student = TinyWrapper(TinyLM().to(device))
teacher = TinyWrapper(TinyLM().to(device))
teacher.model.load_state_dict(student.model.state_dict())
with torch.no_grad():
    teacher.model.lm_head.bias.add_(torch.tensor(
        [0.0, 1.8, -0.4, 0.2, -0.2, 0.1, -0.1], device=device
    ))
teacher.model.eval().requires_grad_(False)

print(f"student parameters: {sum(p.numel() for p in student.model.parameters()):,}")
print(f"student device: {next(student.model.parameters()).device}")

In [ ]:
batch = {
    "input_ids": torch.tensor([[0, 4, 1, 1, 1]], device=device),
    "attention_mask": torch.ones(1, 5, dtype=torch.long, device=device),
    "prompt_lengths": torch.tensor([2], device=device),
}

# The trainer shifts the causal-LM sequence and masks only response labels.
labels = batch["input_ids"][:, 1:]
response_mask = response_token_mask(
    batch["attention_mask"][:, 1:],
    batch["prompt_lengths"] - 1,
    input_ids=labels,
)
print("labels:", labels.tolist())
print("response mask:", response_mask.tolist())

In [ ]:
trainer = OPDTrainer(
    student,
    teacher,
    config=TrainerConfig(
        max_steps=8,
        gradient_accumulation_steps=1,
        learning_rate=0.15,
        weight_decay=0.0,
        use_8bit_optimizer=False,
        max_grad_norm=1.0,
        output_dir=str(REPO_ROOT / "outputs" / "notebook-smoke"),
        checkpoint_every=0,
        seed=7,
    ),
    loss_config=OPDLossConfig(estimator="k3", clamp_log_ratio=10.0),
    optimizer=torch.optim.AdamW(student.model.parameters(), lr=0.15),
)

losses = []
metrics = []
for _ in range(8):
    result = trainer.train_token_batch(batch)
    losses.append(result["loss"])
    metrics.append(result)

print("losses:")
for step, loss in enumerate(losses, start=1):
    print(f"  step {step:02d}: {loss:.12f}")

In [ ]:
plt.figure(figsize=(7, 4))
plt.plot(range(1, len(losses) + 1), losses, marker="o")
plt.xlabel("Optimizer step")
plt.ylabel("veRL-style k3 loss")
plt.title(f"Synthetic OPD loss on {device}")
plt.grid(alpha=0.3)
plt.show()

print(f"initial loss: {losses[0]:.8f}")
print(f"final loss:   {losses[-1]:.8f}")
print(f"overall change: {(losses[-1] - losses[0]) / losses[0] * 100:.2f}%")
print(f"strictly decreasing: {all(a > b for a, b in zip(losses, losses[1:]))}")

In [ ]:
teacher_gradients = [p.grad for p in teacher.model.parameters()]
print("teacher frozen:", all(gradient is None for gradient in teacher_gradients))
print("response tokens per batch:", metrics[-1]["response_tokens"])
print("AMP enabled:", trainer.amp_enabled)
print("AMP dtype:", trainer.active_amp_dtype)
print("FP16 scaler:", trainer.grad_scaler_enabled)
if torch.cuda.is_available():
    print(f"peak allocated: {torch.cuda.max_memory_allocated(device) / 2**20:.1f} MiB")
    print(f"peak reserved:  {torch.cuda.max_memory_reserved(device) / 2**20:.1f} MiB")

## Moving from the smoke test to Qwen3

The guarded cell above delegates to `python -m scripts.real_model_smoke`, which resolves the top-level `configs/` Hydra config and uses `StudentModel`, `TeacherModel`, and `OPDTrainer` for one short real step. With `RUN_REAL_MODEL_SMOKE = False`, opening or executing this notebook never calls `load()` and never downloads model weights.

## Recorded CUDA smoke-test result

Validated separately on an NVIDIA GeForce RTX 4090 with Torch `2.6.0+cu124`:

```text
step 1: 1.4941248894
step 2: 0.1311074346
step 3: 0.1917358190
step 4: 0.1924643069
```

All values were finite and the final loss was below the initial loss. Peak allocated CUDA memory was approximately 17.3 MiB for the tiny model. The curve was not strictly monotonic, which is expected for this small stochastic optimization smoke test.